# 01_setup — Catalog/Schema Setup & Landing Zone Check

Idempotent. Safe to re-run.
1. Create the `bronze`/`silver`/`gold` schemas (and the `fomoless` catalog,
   if it doesn't exist yet — see the note below if that fails)
2. Create the small managed Volume silver's checkpoint lives in
3. Resolve the connection string and confirm it can reach the container
4. Check each of the 9 sources' `raw/<source>/` folder for landed data

**If you don't already have a `fomoless` catalog:** some Databricks
accounts have no Default Storage configured for the metastore, and a bare
`CREATE CATALOG` fails with `Metastore storage root URL does not exist`.
If that happens, it's usually fastest to create the catalog once by hand
in Catalog Explorer (Create → Catalog → Standard, then pick a storage
location) rather than fight Unity Catalog grants from a notebook.

In [0]:
%run ./00_config

# 00_config — Shared Config & Helpers

`%run` by every other notebook. Centralizes the storage connection string,
source list, table/volume names, and small logging/counting helpers.

**This is the connection-string variant of the pipeline.** Bronze reads
files landed in ADLS using the plain Python `azure-storage-blob` SDK and a
storage account connection string — not Auto Loader over an `abfss://`
path, and not a Unity Catalog External Location. That's a deliberate
choice, not a step backward: Databricks Serverless blocks
`spark.conf.set("fs.azure.account.key...")` outright (`CONFIG_NOT_AVAILABLE`),
which is what Auto Loader over `abfss://` would need without an External
Location, and the External Location route needs grants (`READ FILES`,
`WRITE FILES`, `CREATE MANAGED STORAGE`) that may not be available to you
on a shared/class storage account. The Blob SDK never touches Spark's
filesystem config at all -- it's a plain HTTPS call, same mechanism the
live-fetch version (`py_work/`) already uses successfully against public
APIs on serverless -- so it works regardless of either restriction.

**Note on architecture:** Bronze/Silver/Gold are **Unity Catalog managed
tables** — `fomoless.bronze.articles_raw`, `fomoless.silver.articles`,
`fomoless.silver.articles_rejected`, `fomoless.gold.articles`. `01_setup`
creates the schemas (assumes the `fomoless` catalog itself already exists —
see the note in `01_setup` if it doesn't yet).

**Bronze is a plain batch step, not a stream** — since there's no Auto
Loader checkpoint tracking "what's already been read", incrementality
comes from checking each landed file's blob name against what's already in
`bronze.articles_raw` (`source_blob_name`) before downloading it, so a
re-run only pulls genuinely new files. **Silver stays incremental** via its
own Structured Streaming checkpoint, stored in a small Unity Catalog
**managed Volume** (`VOLUME_PATH`, created by `01_setup`) — Databricks-managed
storage under the catalog you already own, so it needs no extra grants
either. That Volume also holds nothing else; dedup-merge audit logs go
straight to ADLS via the same Blob SDK connection as bronze.

## 1. Serverless "spark not initialized" workaround

On some Databricks Serverless sessions, the notebook-provided `spark`
global isn't ready the first time a cell references it, raising
`SystemError: Internal error: spark should be initialized with the first
notebook command.` Explicitly building a session via `databricks.connect`
fixes it and is a no-op anywhere `spark` is already fine.

## 2. Storage connection string

From Azure Portal → your storage account → **Access keys** → Connection
string (key 1 or key 2, either works). Settable via a Databricks widget
(`connection_string`) or the `AZURE_STORAGE_CONNECTION_STRING` environment
variable -- never hardcode it directly in a cell.

## 3. Sources & schema

## 4. Catalog, schemas, tables & volume

## 5. Helpers

[2026-09-22 20:34:20 UTC] Config loaded — container: fomoless, catalog: fomoless


In [0]:
from azure.storage.blob import BlobServiceClient

## 1. Catalog & schemas

In [0]:
existing_catalogs = {row.catalog for row in spark.sql("SHOW CATALOGS").collect()}
if CATALOG_NAME in existing_catalogs:
    log(f"Catalog '{CATALOG_NAME}' already exists — skipping creation")
else:
    try:
        spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
        log(f"Catalog ready: {CATALOG_NAME}")
    except Exception as exc:
        raise EnvironmentError(
            f"Could not create catalog '{CATALOG_NAME}' and it doesn't exist yet. This account likely "
            f"has no default metastore storage -- easiest fix is to create the catalog once by hand in "
            f"Catalog Explorer (Create -> Catalog -> Standard -> pick a storage location), then re-run "
            f"this notebook. Original error: {exc}"
        )

for schema in SCHEMAS.values():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{schema}")
    log(f"Schema ready: {CATALOG_NAME}.{schema}")

[2026-09-22 20:34:24 UTC] Catalog 'fomoless' already exists — skipping creation
[2026-09-22 20:34:24 UTC] Schema ready: fomoless.bronze
[2026-09-22 20:34:24 UTC] Schema ready: fomoless.silver
[2026-09-22 20:34:25 UTC] Schema ready: fomoless.gold


## 2. Managed Volume for silver's checkpoint

In [0]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMAS['silver']}.{VOLUME_NAME}")
log(f"Volume ready: {VOLUME_PATH}")

[2026-09-22 20:34:26 UTC] Volume ready: /Volumes/fomoless/silver/pipeline_internal


## 3. Confirm storage connectivity

Plain Python SDK call over HTTPS using the connection string -- no Unity
Catalog grant needed, so this either works because the connection string
is correct, or fails with a clear auth/network error if it isn't.

In [0]:
container_client = BlobServiceClient.from_connection_string(CONNECTION_STRING).get_container_client(
    CONTAINER_NAME
)
if not container_client.exists():
    raise EnvironmentError(f"Container '{CONTAINER_NAME}' not found — check the connection string.")
log(f"Connected to container: {CONTAINER_NAME}")

[2026-09-22 20:34:27 UTC] Connected to container: fomoless


## 4. Verify landing zone — latest file per source

In [0]:
def latest_blob_in_folder(container_client, folder_prefix: str):
    blobs = list(container_client.list_blobs(name_starts_with=folder_prefix + "/"))
    if not blobs:
        return None, None
    latest = max(blobs, key=lambda b: b.last_modified)
    return latest.name, latest.last_modified


status_rows = []
for src in SOURCES:
    blob_name, last_modified = latest_blob_in_folder(container_client, f'{RAW_PREFIX}/{src["folder"]}')
    if blob_name is None:
        log(f'⚠️  No raw file found yet for {src["folder"]} — bronze will skip it until files land')
        status_rows.append((src["folder"], src["kind"], "MISSING", None, None))
    else:
        status_rows.append((src["folder"], src["kind"], "OK", blob_name, str(last_modified)))

display(
    spark.createDataFrame(
        status_rows, "source STRING, kind STRING, status STRING, latest_blob STRING, last_modified STRING"
    )
)

[2026-09-22 20:34:28 UTC] ⚠️  No raw file found yet for kdnuggets — bronze will skip it until files land
[2026-09-22 20:34:28 UTC] ⚠️  No raw file found yet for guardian — bronze will skip it until files land


source,kind,status,latest_blob,last_modified
devto,json,OK,raw/devto/devto_20260920_030523.json,2026-09-20 03:05:40+00:00
medium,rss,OK,raw/medium/medium_20260920_030550.xml,2026-09-20 03:06:02+00:00
towardsdatascience,rss,OK,raw/towardsdatascience/towardsdatascience_20260920_030612.xml,2026-09-20 03:06:24+00:00
bbc,rss,OK,raw/bbc/bbc_20260920_030632.xml,2026-09-20 03:06:44+00:00
hackernews,json,OK,raw/hackernews/hackernews_20260920_030652.json,2026-09-20 03:07:07+00:00
stackoverflow_blog,rss,OK,raw/stackoverflow_blog/stackoverflow_blog_20260920_030715.xml,2026-09-20 03:07:29+00:00
stackexchange,json,OK,raw/stackexchange/stackexchange_20260920_030739.json,2026-09-20 03:07:52+00:00
kdnuggets,rss,MISSING,null,null
guardian,json,MISSING,null,null


In [0]:
sources_with_data = sum(1 for row in status_rows if row[2] == "OK")
log(f"{sources_with_data} of {len(SOURCES)} sources have landed data.")

if sources_with_data == 0:
    raise FileNotFoundError(
        f"No raw files found for any source under '{RAW_PREFIX}/' in container '{CONTAINER_NAME}'. "
        "Land files there first, or confirm CONTAINER_NAME in 00_config points at the right container."
    )

[2026-09-22 20:34:36 UTC] 7 of 9 sources have landed data.
